# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s.

**Note:** All entities are referenced by their `@id` as per best practice.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.record_sets
print("Available RecordSets:")
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', '')}")
    print("  fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {getattr(field, 'name', '')}")
    print()

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis.

All references are made using their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded: {rs_id}, shape: {df.shape}")

# Show columns of first available record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing: filter records, normalize numeric fields, and group/categorize data using `@id` references.

In [ ]:
# Select the first record set for demonstration
if not record_set_ids:
    raise ValueError("No record sets found in dataset.")
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# List numeric fields
numeric_field_id = None
for field in dataset.get_record_set(record_set_id).fields:
    dtype = getattr(field, 'data_type', '').lower()
    if dtype in {'float', 'integer', 'number'}:
        numeric_field_id = field.id
        print(f"Using numeric field: {field.name} (@id: {field.id}) - type: {field.data_type}")
        break

if numeric_field_id is None:
    print("No numeric fields detected; EDA will be limited.")
else:
    # Show field for grouping (first non-numeric field)
    group_field_id = None
    for field in dataset.get_record_set(record_set_id).fields:
        if field.id != numeric_field_id and getattr(field, 'data_type', '').lower() not in {'float', 'integer', 'number'}:
            group_field_id = field.id
            print(f"Grouping by field: {field.name} (@id: {field.id})")
            break

    # Clean column names (replace special chars, as sometimes @id is a URI)
    cleaned_cols = {col: col.split('/')[-1] for col in df.columns}
    df = df.rename(columns=cleaned_cols)

    # Use last segment of @id for columns (helps with plotting)
    num_col = numeric_field_id.split('/')[-1] if '/' in numeric_field_id else numeric_field_id
    group_col = group_field_id.split('/')[-1] if group_field_id and '/' in group_field_id else group_field_id

    # Convert to numeric (handle errors)
    df[num_col] = pd.to_numeric(df[num_col], errors='coerce')

    threshold = df[num_col].mean() if pd.notnull(df[num_col]).any() else 0
    filtered_df = df[df[num_col] > threshold]
    print(f"Filtered records with {num_col} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize column
    if filtered_df[num_col].std() > 0:
        filtered_df[num_col + "_normalized"] = (filtered_df[num_col] - filtered_df[num_col].mean()) / filtered_df[num_col].std()
    else:
        filtered_df[num_col + "_normalized"] = filtered_df[num_col]

    print(f"\nNormalized {num_col} for filtered records:")
    display(filtered_df[[num_col, num_col + '_normalized']].head())

    if group_col and group_col in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_col)[num_col].mean().reset_index()
        print(f"Grouped mean {num_col} by {group_col}:")
        display(grouped_df.head())

## 5. Visualization

Visualize distributions and relationships using pandas and matplotlib. All plots use fields by their `@id` or last path segment.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[num_col].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {num_col}")
    plt.xlabel(num_col)
    plt.ylabel("Count")
    plt.show()

    if group_col and group_col in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_col, y=num_col, data=df)
        plt.title(f"{num_col} by {group_col}")
        plt.xlabel(group_col)
        plt.ylabel(num_col)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we used the Croissant schema and `mlcroissant` to load, explore, and process a FAIR-compliant dataset. 

- Entities and data were referenced by `@id` for reliability and reproducibility.  
- We demonstrated record set loading, overview, transformation, and basic visualization.
- The structure supports further analysis or automated reproducible ML workflows using Croissant metadata.

Refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/clients/python/) for more advanced usage.